In [29]:
import boto3
import fsspec 
import shutil
import os
from virtualizarr import open_virtual_dataset
from virtualizarr.readers.hdf import HDFVirtualBackend
import xarray as xr
import helpers

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Todos for this notebook

- [x] DRY up some functions, perhaps separate to another file
- [x] Extract tests for kerchunk and hdfreader and encoding to appendices
- [ ] Come up with a function for validation
- [ ] Execute as S3 storage
- [ ] Estimate time to complete 2004-2024

Nice to have:
- [ ] Add arraylake option

# Initialize the store

## function, local, s3 or arraylake

In [21]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster()
client = Client(cluster)
cluster.scale(8)

In [25]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/abarciauskas-bgse/proxy/8787/status,
Dashboard: /user/abarciauskas-bgse/proxy/8787/status,Workers: 8
Total threads: 32,Total memory: 121.25 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36103,Workers: 8
Dashboard: /user/abarciauskas-bgse/proxy/8787/status,Total threads: 32
Started: Just now,Total memory: 121.25 GiB
Comm: tcp://127.0.0.1:36157,Total threads: 4
Dashboard: /user/abarciauskas-bgse/proxy/34003/status,Memory: 15.16 GiB
Nanny: tcp://127.0.0.1:44763,


In [4]:
#if overwriting the s3 store, you need to run the following lines
#!pip install awscli
#!aws s3 rm --recursive s3://nasa-veda-scratch/icechunk/{store_name}
store = helpers.find_or_create_icechunk_store(store_name="MUR-JPL-L4-GLOB-v4.1-virtual", store_type="local", overwrite=False)
store

In [5]:
fs = fsspec.filesystem("s3", anon=False)

## Create initial store with data from 2002

Open a few days of data and virtualize + concatenize datasets with dmrpp

In [ ]:
mur_sst_files_2002 = helpers.list_mur_sst_files(date_prefix="2002", fs=fs)
mur_sst_dmrpps_2002 = [f + '.dmrpp' for f in mur_sst_files_2002]
virtual_ds_2002 = helpers.create_virtual_ds(dmrpps=mur_sst_dmrpps_2002)

## Write to icechunk

In [ ]:
%%time
virtual_ds_2002.virtualize.to_icechunk(store)
store.commit("Wrote 2002 data")

## Append 2003

One file in 2003 had a different encoding, so the the list of 2003 files is split into 3 lists. See 2a for how problematic date was found. All dates apart from the date with the different encoding are written as virtual stores. The problematic data is written as zarr.

See and run `helpers.get_codecs` with a list of virtual datasets to check all codecs are the same.

### 1. List files

In [6]:
problematic_date = "20030911090000"
mur_sst_files = helpers.list_mur_sst_files(date_prefix="2003")
problematic_file = helpers.list_mur_sst_files(date_prefix="20030911090000")
problematic_file

['s3://podaac-ops-cumulus-protected/MUR-JPL-L4-GLOB-v4.1/20030911090000-JPL-L4_GHRSST-SSTfnd-MUR-GLOB-v02.0-fv04.1.nc']

In [8]:
mur_sst_files_2003_1, mur_sst_files_2003_diff_enc, mur_sst_files_2003_2 = helpers.split_list(mur_sst_files, problematic_file[0])

### 2. Open + write first set of files as virtual datasets using the DMRPP reader

Skip this step if data is written.

In [ ]:
mur_sst_files_2003_1_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2003_1]
virtual_ds_2003_1 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2003_1_dmrpps)

In [ ]:
virtual_ds_2003_1.virtualize.to_icechunk(store, append_dim='time')
store.commit("Wrote first part of 2003 data")

### 3. Write data with different encoding as zarr.

Skip this step if data is written.

In [ ]:
# this takes about a minute and a lot of memory.
ds = xr.open_dataset(fs.open(problematic_file[0]))
ds.to_zarr(store, append_dim='time')
store.commit(f"Wrote {problematic_date} in zarr")

### 4. Write the rest of 2003 as virtual data

In [9]:
mur_sst_files_2003_2_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2003_2]
virtual_ds_2003_2 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2003_2_dmrpps)

111

In [12]:
virtual_ds.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote {problematic_date to end of 2003 as virtual data.")

'NZJFXTW7DH5RJQ6GDQA0'

# Validate

In [13]:
lat_slice = slice(42,43)
lon_slice = slice(-122,-121)
time_slice = slice('2003-10-01', '2003-10-11')

In [14]:
xds = xr.open_zarr(store)
xds

/tmp/ipykernel_83/1944499603.py:1: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  xds = xr.open_zarr(store)


<xarray.Dataset> Size: 11TB
Dimensions:           (time: 579, lat: 17999, lon: 36000)
Coordinates:
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
  * time              (time) datetime64[ns] 5kB 2002-06-01T09:00:00 ... 2003-...
Data variables:
    mask              (time, lat, lon) float32 2TB dask.array<chunksize=(1, 1447, 2895), meta=np.ndarray>
    analysed_sst      (time, lat, lon) float64 3TB dask.array<chunksize=(1, 1023, 2047), meta=np.ndarray>
    sea_ice_fraction  (time, lat, lon) float64 3TB dask.array<chunksize=(1, 1447, 2895), meta=np.ndarray>
    analysis_error    (time, lat, lon) float64 3TB dask.array<chunksize=(1, 1023, 2047), meta=np.ndarray>
Attributes: (12/47)
    Conventions:                CF-1.5
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    MUR = "Multi-scale Ultra-high Reolution"
    creator_email:              ghrsst@podaac.jpl.nasa.gov
    ...                         ...
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    time_coverage_end:          20030912T210000Z
    time_coverage_start:        20030911T210000Z
    title:                      Daily MUR SST, Final product
    uuid:                       27665bc0-d5fc-11e1-9b23-0800200c9a66
    westernmost_longitude:      -180.0

In [15]:
%%time
xds.analysed_sst.sel(lat=lat_slice, lon=lon_slice, time=time_slice).mean().values

CPU times: user 401 ms, sys: 34.8 ms, total: 436 ms
Wall time: 2.39 s


array(279.63857298)

In [18]:
%%time
og_ds = xr.open_mfdataset([fs.open(f) for f in mur_sst_files_1], parallel=True)

CPU times: user 3.15 s, sys: 299 ms, total: 3.44 s
Wall time: 27.5 s


In [19]:
%%time
og_ds.analysed_sst.sel(lat=lat_slice, lon=lon_slice, time=time_slice).mean().values

CPU times: user 771 ms, sys: 74.4 ms, total: 846 ms
Wall time: 8.69 s


array(279.63857298)